In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import joblib
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis.client_config import ClientConfig
from meridian.analysis.helper import GCPClient

In [ ]:
# Cargar 'model.pkl'
model = "model.pkl"
mmm = joblib.load(model)

In [ ]:
# Congiguración personalizada para el cliente
client_config = ClientConfig(
    config_path="./config.yaml"
)

In [ ]:
# Esta celda se debe ejecutar para subir los reportes a GCS y cargar datos a BigQuery.
# Si esto no es necesario, se puede omitir. --- IGNORE ---

import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "path/to/your/service_account_key.json"
gcp_client = GCPClient()

In [ ]:
mmm_summarizer = summarizer.Summarizer(mmm, client_config)
start_date_ = "2025-01-01"
end_date_ = "2025-09-01"

mmm_summarizer.output_model_results_summary(
    filename="mmm_report.html",
    filepath="./reports/Auto/",
    start_date=start_date_,
    end_date=end_date_,
)

# Guardar en GCS
gcp_client.upload_file_to_gcs(
    bucket_name="your-gcs-bucket-name",
    prefix='Reports/Auto',
    full_path="./reports/Auto/mmm_report.html",
)

In [ ]:
mmm_summarizer = summarizer.Summarizer(mmm, client_config)
start_date_ = "2025-01-01"
end_date_ = "2025-09-01"
start_date_cm_ = "2024-01-01"
end_date_cm_ = "2024-09-01"

mmm_summarizer.output_comparison_metrics_summary(
    filename="mmm_report_comparison_metrics.html",
    filepath="./reports/Auto/",
    start_date=start_date_,
    end_date=end_date_,
    start_date_cm=start_date_cm_,
    end_date_cm=end_date_cm_,
    bq_product_or_service="Auto"
)

# Guardar en GCS
gcp_client.upload_file_to_gcs(
    bucket_name="your-gcs-bucket-name",
    prefix='Reports/Auto',
    full_path="./reports/Auto/mmm_report_comparison_metrics.html",
)

# Cargar en BigQuery
for file in os.listdir("./reports/Auto/"):
    if file.endswith(".parquet"):
        parquet_path = os.path.join("./reports/Auto/", file)
        gcp_client.load_parquet_to_bq(
            parquet_path=parquet_path,
            table_id="your-project-id.your_dataset.your_table",
        )

In [ ]:
budget_optimizer = optimizer.BudgetOptimizer(mmm, client_config)
optimization_results = budget_optimizer.optimize()

optimization_results.output_optimization_summary(
    filename="optimization_output.html",
    filepath="./reports/Auto",
)

# Guardar en GCS
gcp_client.upload_file_to_gcs(
    bucket_name="your-gcs-bucket-name",
    prefix='Reports/Auto/',
    full_path="./reports/Auto/optimization_output.html",
)

In [ ]:
from meridian.model.model import save_mmm
save_mmm(mmm, "./model.pkl")